<a href="https://colab.research.google.com/github/alirezaataei1375-eng/Convergence-rate-QLLG-Bear-2/blob/main/Large_Spin_QLLG_to_LLG_final%2C_Bear_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
! pip install qutip
# ==========================================================
# IMPORTS
# ==========================================================
import numpy as np
import qutip as qt
import matplotlib.pyplot as plt

# ==========================================================
# PARAMETERS
# ==========================================================
Bx, By, Bz = 1/3, 2/3, 2/3
gamma = 1.0
kappa = 0.3
J = 1.0

hbar = 1.0
dt = 5e-2
steps = int(2 / dt)

h_values = np.linspace(0, 2, 25)
S_values = np.arange(0.5, 40.5, 0.5)

# ==========================================================
# CLASSICAL DYNAMICS
# ==========================================================
gamma_eff = 1 / (1 + kappa**2)

def llg_rhs(n, B):
    return (
        -gamma_eff * np.cross(n, B)
        - kappa * gamma_eff * np.cross(n, np.cross(n, B))
    )

def rk4_step_pair(n1, n2, Bext):
    def f1(a,b): return llg_rhs(a, gamma*Bext/(S*hbar) - (J/(S*hbar)**2)*b)
    def f2(a,b): return llg_rhs(b, gamma*Bext/(S*hbar) - (J/(S*hbar)**2)*a)

    k1_1 = f1(n1,n2); k1_2 = f2(n1,n2)
    k2_1 = f1(n1+0.5*dt*k1_1, n2+0.5*dt*k1_2)
    k2_2 = f2(n1+0.5*dt*k1_1, n2+0.5*dt*k1_2)
    k3_1 = f1(n1+0.5*dt*k2_1, n2+0.5*dt*k2_2)
    k3_2 = f2(n1+0.5*dt*k2_1, n2+0.5*dt*k2_2)
    k4_1 = f1(n1+dt*k3_1, n2+dt*k3_2)
    k4_2 = f2(n1+dt*k3_1, n2+dt*k3_2)

    n1n = n1 + dt*(k1_1+2*k2_1+2*k3_1+k4_1)/6
    n2n = n2 + dt*(k1_2+2*k2_2+2*k3_2+k4_2)/6

    def norm(v):
        nrm = np.linalg.norm(v)
        return v / nrm if nrm > 1e-12 else v

    return norm(n1n), norm(n2n)

# ==========================================================
# STORAGE OVER S
# ==========================================================
results = {}

# ==========================================================
# LOOP OVER S
# ==========================================================
for S in S_values:

    dim = int(2*S + 1)

    # spin operators
    Sx = hbar * qt.jmat(S, 'x')
    Sy = hbar * qt.jmat(S, 'y')
    Sz = hbar * qt.jmat(S, 'z')

    I = qt.qeye(dim)

    def op1(op): return qt.tensor(op, I)
    def op2(op): return qt.tensor(I, op)

    Sx1, Sy1, Sz1 = op1(Sx), op1(Sy), op1(Sz)
    Sx2, Sy2, Sz2 = op2(Sx), op2(Sy), op2(Sz)

    I_tot = qt.tensor(I, I)

    # ======================================================
    # OBSERVABLES
    # ======================================================
    def local_spins(state):
        s1 = np.array([
            qt.expect(Sx1, state),
            qt.expect(Sy1, state),
            qt.expect(Sz1, state)
        ])
        s2 = np.array([
            qt.expect(Sx2, state),
            qt.expect(Sy2, state),
            qt.expect(Sz2, state)
        ])
        return s1, s2

    def correlation_matrix(state):
        ops1 = [Sx1, Sy1, Sz1]
        ops2 = [Sx2, Sy2, Sz2]

        C = np.zeros((3,3))
        for a in range(3):
            for b in range(3):
                C[a,b] = np.real(
                    qt.expect(ops1[a]*ops2[b], state)
                    - qt.expect(ops1[a], state) * qt.expect(ops2[b], state)
                )
        return C

    def covariance_spin1(state):
        ops = [Sx1, Sy1, Sz1]

        m = np.array([
            qt.expect(Sx1,state),
            qt.expect(Sy1,state),
            qt.expect(Sz1,state)
        ])

        G = np.zeros((3,3))
        for a in range(3):
            for b in range(3):
                G[a,b] = np.real(
                    qt.expect(ops[a]*ops[b] + ops[b]*ops[a], state)
                    - 2*m[a]*m[b]
                )
        return G

    def build_hamiltonian(Bext):
        return (
            (J/(S* hbar)**2)*(Sx1*Sx2 + Sy1*Sy2 + Sz1*Sz2)
            - (gamma/(S*hbar))*(Bext[0]*(Sx1+Sx2)
                       + Bext[1]*(Sy1+Sy2)
                       + Bext[2]*(Sz1+Sz2))
        )

    def qllg_rhs(state, H):
        E = qt.expect(H, state)
        H_eff = H - E * I_tot
        coeff = (-1j * (1 - 1j*kappa)) / (hbar * (1 + kappa**2))
        return coeff * (H_eff * state)

    def qllg_step(state, H):
        return (state + dt * qllg_rhs(state, H)).unit()

    # ======================================================
    # STORAGE per S
    # ======================================================
    max_angle1_list = []
    max_angle2_list = []
    max_entropy_list = []
    max_corr_list = []
    max_len_def1_list = []
    max_len_def2_list = []
    max_closure_list = []

    # ======================================================
    # LOOP OVER h
    # ======================================================
    for h in h_values:

        Bext = h * np.array([Bx, By, Bz])
        H = build_hamiltonian(Bext)

        psi1 = qt.spin_coherent(S, np.pi/3, np.pi * 0.2)
        psi2 = qt.spin_coherent(S, np.pi/2, np.pi * 0.1)
        psi = qt.tensor(psi1, psi2)

        def norm(v):
            n = np.linalg.norm(v)
            return v / n if n > 1e-12 else v

        s1, s2 = local_spins(psi)
        n1 = norm(s1)
        n2 = norm(s2)

        angle1, angle2 = [], []
        entropy, corr = [], []
        len_def1, len_def2 = [], []
        closure = []

        # ==================================================
        # TIME EVOLUTION
        # ==================================================
        for _ in range(steps):

            psi = qllg_step(psi, H)

            s1_q, s2_q = local_spins(psi)

            nq1 = norm(s1_q)
            nq2 = norm(s2_q)

            n1, n2 = rk4_step_pair(n1, n2, Bext)

            angle1.append(np.arccos(np.clip(np.dot(nq1, n1), -1, 1)))
            angle2.append(np.arccos(np.clip(np.dot(nq2, n2), -1, 1)))

            rho = psi.proj()
            rho1 = qt.ptrace(rho, 0)
            eig = np.real(rho1.eigenenergies())
            eig = eig[eig > 1e-12]
            entropy.append(-np.sum(eig * np.log(eig + 1e-12)))

            C = correlation_matrix(psi)
            corr.append(np.linalg.norm(C) / S**2)

            # ✔ FIXED: BOTH LENGTH ERRORS
            len_def1.append(np.abs(1 - np.linalg.norm(s1_q)/(S*hbar)))
            len_def2.append(np.abs(1 - np.linalg.norm(s2_q)/(S*hbar)))

            G = covariance_spin1(psi)
            nq = nq1
            Gcoh = hbar * S * (np.eye(3) - np.outer(nq, nq))
            closure.append(np.linalg.norm(G - Gcoh)/(hbar*S))

        # store max values
        max_angle1_list.append(np.max(angle1))
        max_angle2_list.append(np.max(angle2))
        max_entropy_list.append(np.max(entropy))
        max_corr_list.append(np.max(corr))
        max_len_def1_list.append(np.max(len_def1))
        max_len_def2_list.append(np.max(len_def2))
        max_closure_list.append(np.max(closure))

    results[S] = {
        "h": h_values,
        "angle1": max_angle1_list,
        "angle2": max_angle2_list,
        "entropy": max_entropy_list,
        "corr": max_corr_list,
        "len1": max_len_def1_list,
        "len2": max_len_def2_list,
        "closure": max_closure_list
    }

# ==========================================================
# PLOTS (example: final S dependence)
# ==========================================================
S_list = np.array(list(results.keys()))
eps = 0.02
# ==========================================================
# 1. LENGTH DEFECT (CLEAR SEPARATION)
# ==========================================================
plt.figure()

y1 = [np.max(results[S]["len1"])*100 for S in S_list]
y2 = [np.max(results[S]["len2"])*100 for S in S_list]

plt.plot(S_list - eps, y1,
         marker="o", linestyle="-",
         linewidth=2, alpha=0.9,
         label="Spin 1")

plt.plot(S_list + eps, y2,
         marker="s", linestyle="--",
         linewidth=2, alpha=0.9,
         label="Spin 2")

plt.xlabel("S",fontsize=16)
plt.ylabel("Length error (%)",fontsize=16)
plt.xticks(fontsize=14)
plt.yticks(fontsize=14)
plt.legend(fontsize=14)
plt.tight_layout()
#plt.grid(alpha=0.3)
plt.show()

# ==========================================================
# 2. ANGLE ERROR (STRONG CONTRAST STYLE)
# ==========================================================
plt.figure()

y1 = [np.max(results[S]["angle1"]) for S in S_list]
y2 = [np.max(results[S]["angle2"]) for S in S_list]

plt.plot(S_list - eps, y1,
         marker="^", linestyle="-",
         linewidth=2, alpha=0.9,
         label="Spin 1")

plt.plot(S_list + eps, y2,
         marker="v", linestyle="--",
         linewidth=2, alpha=0.9,
         label="Spin 2")

plt.xlabel("S",fontsize=16)
plt.ylabel("Angle error",fontsize=16)
#plt.grid(alpha=0.3)
plt.xticks(fontsize=14)
plt.yticks(fontsize=14)
plt.legend(fontsize=14)
plt.tight_layout()
plt.show()

# ==========================================================
# 3. ENTANGLEMENT (SINGLE CURVE)
# ==========================================================
plt.figure()

y = [np.max(results[S]["entropy"]) for S in S_list]

plt.plot(S_list, y,
         marker="o", linewidth=2)

plt.xlabel("S",fontsize=16)
plt.ylabel("Entanglement entropy",fontsize=16)
#plt.grid(alpha=0.3)
plt.xticks(fontsize=14)
plt.yticks(fontsize=14)
plt.tight_layout()
plt.show()

# ==========================================================
# 4. CORRELATION NORM
# ==========================================================
plt.figure()

y = [np.max(results[S]["corr"]) for S in S_list]

plt.plot(S_list, y,
         marker="o", linewidth=2)

plt.xlabel("S",fontsize=16)
plt.ylabel("Correlation norm",fontsize=16)
#plt.grid(alpha=0.3)
plt.xticks(fontsize=14)
plt.yticks(fontsize=14)
plt.tight_layout()
plt.show()

KeyboardInterrupt: 